# Anti-DPO: Google Colab training pilot

Use a Colab GPU runtime. The first run is limited to 50 optimizer steps to verify memory use, loss direction, and checkpointing. The committed `data/russian_qa` is used directly; run the preparation notebook only when changing preparation parameters.

In [ ]:
# Colab setup. Select Runtime > Change runtime type > T4 GPU before running.
REPO_URL = 'https://github.com/moliksq/Mauvais.git'
REPO_DIR = '/content/Mauvais'
!rm -rf $REPO_DIR
!git clone --depth 1 $REPO_URL $REPO_DIR
%cd $REPO_DIR/anti_dpo_experiment
!pip -q install -U 'transformers>=4.51,<5' 'trl>=0.16,<0.20' 'peft>=0.14' 'accelerate>=1.3' 'datasets>=3.0' 'matplotlib>=3.8'
!nvidia-smi

In [ ]:
from pathlib import Path
import torch

ROOT = Path.cwd()
assert torch.cuda.is_available(), 'No GPU runtime. Enable a Colab GPU and rerun.'
assert (ROOT / 'data' / 'russian_qa' / 'dataset_dict.json').is_file()
print('GPU:', torch.cuda.get_device_name(0))
print('Dataset path:', ROOT / 'data' / 'russian_qa')

In [ ]:
import subprocess, sys

config = {
    'max_steps': 50, 'batch_size': 1, 'gradient_accumulation_steps': 8,
    'max_length': 768, 'max_prompt_length': 256, 'learning_rate': 5e-6,
    'beta': .1, 'lora_r': 16, 'lora_alpha': 32,
}
command = [sys.executable, 'scripts/train.py', '--dataset_path', 'data/russian_qa', '--output_dir', 'outputs/anti_dpo_pilot', *sum(([f'--{key}', str(value)] for key, value in config.items()), [])]
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
from pathlib import Path

checkpoints = sorted(Path('outputs/anti_dpo_pilot').glob('checkpoint-*'))
print('Checkpoints:', checkpoints)
!du -sh outputs/anti_dpo_pilot

## LR-LoRA feasibility probe

The project includes the learnable-sinc LR-LoRA implementation from the supplied paper. It is intentionally separate from the standard PEFT training loop until the LoRA pilot is stable.

In [ ]:
import sys, torch
from torch import nn
sys.path.insert(0, str(ROOT / 'src'))
from lr_lora import LearnableRankLoRALinear

probe = LearnableRankLoRALinear(nn.Linear(64, 64), rank=8, alpha=16, num_basis=8).cuda()
with torch.no_grad():
    probe.nonlinearity.amplitudes.uniform_(-.1, .1)
print('LR-LoRA stable rank:', float(probe.stable_rank()))
print('Trainable parameters:', sum(p.numel() for p in probe.parameters() if p.requires_grad))